# GRPO: from reward to an update

Group-relative advantages, policy ratios, and clipping.

> All examples run on CPU without model downloads and use synthetic data. They explain mechanisms; real training quality must be evaluated separately.

## What to understand

GRPO compares several responses to the same prompt. Rewards are normalized within each group to form advantages. The policy ratio compares the current policy with the sampling policy; clipping limits part of the update.

A real trainer also needs generation, token masks, saved old log-probs, reference/KL when used, and a validated reward. This exercise isolates the clipped objective without KL or generation. It neither replaces Studio’s trainer nor runs TRL. Constant rewards must not produce NaNs.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Run the experiment

Run cells in order. Edit the code; Shift + Enter runs the current cell.

In [ ]:
rewards = torch.tensor([[0., 1., 2., 3.], [1., 1., 1., 1.]])
advantages = (rewards - rewards.mean(-1, keepdim=True)) / rewards.std(-1, keepdim=True, unbiased=False).clamp_min(1e-6)
logits = nn.Parameter(torch.zeros(2, 4))
old_logp = logits.detach().log_softmax(-1).clone()
optimizer = torch.optim.Adam([logits], lr=0.08)
for step in range(8):
    logp = logits.log_softmax(-1)
    ratio = (logp - old_logp).exp()
    objective = torch.minimum(ratio * advantages, ratio.clamp(0.8, 1.2) * advantages)
    loss = -objective.mean()
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
print('advantages:', advantages)
print('new probabilities:', logits.softmax(-1).detach())
assert torch.isfinite(loss) and torch.equal(advantages[1], torch.zeros(4))

## Try it yourself

Change the rewards, including constant groups. Which answers gain probability? Explain why old_logp must not be recomputed after each optimizer step.

Save a notebook copy and record what you changed, expected, and measured.

## Continue in Studio

The workspace link above the lesson opens related tools. This notebook does not launch large training jobs. Before real training, check data, resources, and method support for the chosen model.

## Sources

- [DeepSeekMath / GRPO](https://arxiv.org/abs/2402.03300)